# SafeSight gait classifier

Train the walking-versus-still label that `pi/runtime.py` uses to scale time-to-contact.

The Pi does not run scikit-learn. This notebook fits a logistic regression, then writes the weights, the feature mean, and the feature scale as JSON. `pi/gait.py` scores that file with a dot product and a sigmoid. The feature code imported below is the same module the Pi runs, so the vector cannot drift between train and inference.

`pc/PIPELINE.md` describes the rest of the wearable loop (geometry, tone, MQTT). This notebook is only the gait model.

Run the cells from the top. The kernel's working directory should be `pc/` or the repo root.


In [1]:
import json
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

HERE = Path.cwd().resolve()
if (HERE / "train_gait.ipynb").is_file():
    REPO = HERE.parent
elif (HERE / "pc" / "train_gait.ipynb").is_file():
    REPO = HERE
else:
    raise SystemExit("Open this notebook with the working directory set to pc/ or the repo root.")

sys.path.insert(0, str(REPO / "pi"))
from gait import FEATURE_NAMES, WINDOW_S, GaitModel, features_from_samples
from session_io import JsonlWriter, load_records

print("repo", REPO)
print("window", WINDOW_S, "s  features", ", ".join(FEATURE_NAMES))


repo C:\Users\jacob\Documents\UNT\SmartSight\SafeSight
window 1.5 s  features mag_mean, mag_std, mag_ptp, x_std, y_std, z_std, zcr_hz, band_energy


## Load collection files

Set `PATHS` to one or more JSONL files from `collect_data.py`. Leave it empty to build a synthesized still-then-walking session. That demo is only here so the notebook runs before a real collection. It is not a model of a wearer.

Files from different Pi boots do not share a monotonic clock. Windows are built inside each file, then concatenated.


In [ ]:
# Collected sessions. Empty builds data/demo_session.jsonl and trains on that.
PATHS = [
    "C:\\Users\\jacob\\Documents\\UNT\\SmartSight\\SafeSight\\data/session_20261111_224354.jsonl",
    "C:\\Users\\jacob\\Documents\\UNT\\SmartSight\\SafeSight\\data/session_20261111_225412.jsonl"
]

HOP_S = 0.5
GAIT_LABELS = ("walking", "still")


def synthesize(path):
    """Quiet standing, then a 2 Hz walk. Obstacle tags are present and ignored by the gait label."""
    path = Path(path)
    # JsonlWriter appends. A second run would stack another copy on the same clock.
    if path.exists():
        path.unlink()
    writer = JsonlWriter(str(path))
    writer.write({
        "rec": "meta",
        "mono": 0.0,
        "ts": 0.0,
        "imu_hz": 50,
        "tof_hz": 15,
        "resolution": "8x8",
        "note": "Synthesized demo. Replace PATHS with a real collect_data.py file.",
    }, force_flush=True)
    hz = 50.0
    duration = 32.0
    for i in range(int(duration * hz)):
        t = i / hz
        if t < 16.0:
            # Slow sway, well below the step band, so a quiet stand
            # does not look like a walk just because of sensor hiss.
            x = 0.04 * math.sin(2.0 * math.pi * 0.20 * t)
            y = 0.03 * math.sin(2.0 * math.pi * 0.15 * t + 0.4)
            z = 9.806 + 0.05 * math.sin(2.0 * math.pi * 0.25 * t)
        else:
            w = 2.0 * math.pi * 2.0 * t
            x = 1.6 * math.sin(w)
            y = 0.5 * math.sin(w + 0.8)
            z = 9.806 + 2.6 * math.sin(w)
        writer.write({
            "rec": "imu",
            "mono": round(t, 4),
            "ts": round(t, 3),
            "x": round(x, 4),
            "y": round(y, 4),
            "z": round(z, 4),
            "fake": True,
        })
    # One ToF line so the file has both sensors. Training does not read it.
    writer.write({
        "rec": "tof",
        "mono": 1.0,
        "ts": 1.0,
        "resolution": "8x8",
        "distances": [2000] * 64,
        "status": [5] * 64,
        "silicon_temp_c": None,
        "fake": True,
    })
    for mono, label in ((0.0, "still"), (8.0, "obstacle"), (16.0, "walking"), (24.0, "arm-wave")):
        writer.write({"rec": "event", "mono": mono, "ts": mono, "label": label}, force_flush=True)
    writer.close()
    return path


def label_for(t0, t1, events):
    """Gait label in force at t0, or None when a gait event falls inside the window."""
    active = None
    for ts, label in events:
        if ts <= t0 + 1e-6:
            active = label
        elif ts < t1 - 1e-3:
            return None
        else:
            break
    return active


def windows_from_records(records, hop_s=HOP_S):
    imu = sorted(
        (row["mono"], row["x"], row["y"], row["z"])
        for row in records
        if row.get("rec") == "imu"
    )
    events = sorted(
        (row["mono"], row["label"])
        for row in records
        if row.get("rec") == "event" and row.get("label") in GAIT_LABELS
    )
    if len(imu) < 2:
        return []
    found = []
    t0 = imu[0][0]
    t_last = imu[-1][0]
    while t0 + WINDOW_S <= t_last + 1e-6:
        t1 = t0 + WINDOW_S
        chunk = [row for row in imu if t0 - 1e-9 <= row[0] <= t1 + 1e-9]
        found.append({
            "t0": t0,
            "t1": t1,
            "label": label_for(t0, t1, events),
            "features": features_from_samples(chunk),
            "samples": chunk,
        })
        t0 += hop_s
    return found


paths = [Path(p) for p in PATHS]
if not paths:
    demo = REPO / "data" / "demo_session.jsonl"
    synthesize(demo)
    paths = [demo]
    print("PATHS is empty. Training on the synthesized demo:", demo)
else:
    print("Training on", ", ".join(str(p) for p in paths))

windows = []
for path in paths:
    file_windows = windows_from_records(list(load_records(path)))
    for item in file_windows:
        item["source"] = str(path)
    windows.extend(file_windows)
    print("%s  imu-windows %d" % (path.name, len(file_windows)))

for index, item in enumerate(windows):
    item["index"] = index
labeled_now = sum(1 for item in windows if item["label"] in GAIT_LABELS and item["features"] is not None)
print("windows %d   labeled from events %d" % (len(windows), labeled_now))


Training on data\session_20261111_224354.jsonl, data\session_20261111_225412.jsonl


FileNotFoundError: [Errno 2] No such file or directory: 'data\\session_20261111_224354.jsonl'

## Labels

Event flags are the label. `walking` and `still` apply from that moment until the next gait flag. A window that contains a change of gait is dropped. `obstacle`, `step-down`, and `arm-wave` stay in the file for later use and do not change this label.

If the flags are sparse, fill `MANUAL_LABELS` with `{window_index: "walking"}` or `"still"` and run this cell again. The index is the one printed in the table. When `ipywidgets` is installed, the same windows show up as dropdowns.


In [ ]:
# Optional hand labels. Keys are the window indexes printed below.
MANUAL_LABELS = {}


def apply_manual(windows, manual):
    for key, label in manual.items():
        if label not in GAIT_LABELS:
            raise ValueError("MANUAL_LABELS[%s] must be walking or still, got %r" % (key, label))
        windows[int(key)]["label"] = label


apply_manual(windows, MANUAL_LABELS)

unlabeled = [
    item for item in windows
    if item["features"] is not None and item["label"] not in GAIT_LABELS
]
labeled = [
    item for item in windows
    if item["features"] is not None and item["label"] in GAIT_LABELS
]
print("labeled %d   unlabeled with features %d" % (len(labeled), len(unlabeled)))

counts = {name: sum(1 for item in labeled if item["label"] == name) for name in GAIT_LABELS}
print("class counts", counts)
need_manual = min(counts.values()) < 4
if need_manual and unlabeled:
    print("index   t0      mag_std   band_energy   auto")
    for item in unlabeled[:40]:
        feats = item["features"]
        print("%5d  %7.2f  %8.3f  %12.3f   %s" % (
            item["index"], item["t0"], feats[1], feats[7], item["label"],
        ))
    if len(unlabeled) > 40:
        print("... %d more" % (len(unlabeled) - 40))
    try:
        import ipywidgets as widgets
        from IPython.display import display
    except Exception:
        widgets = None
    if widgets is not None:
        boxes = []
        for item in unlabeled[:40]:
            boxes.append(widgets.Dropdown(
                options=["", "walking", "still"],
                value="",
                description=str(item["index"]),
            ))
        display(widgets.VBox(boxes))
        print("Pick labels, then copy them into MANUAL_LABELS and run this cell again.")
    else:
        print("Set MANUAL_LABELS and run this cell again. ipywidgets is not installed.")
elif unlabeled:
    print("%d windows contain a gait change and stay unlabeled." % len(unlabeled))
if need_manual:
    raise RuntimeError(
        "Need at least 4 windows of walking and 4 of still. "
        "Add event flags in the collection file or fill MANUAL_LABELS."
    )


## Fit and export

Features are standardized inside the fit (`StandardScaler`, divide by N). The split is by time inside each class. Test windows that overlap the end of the training slice are dropped.

The JSON written to `pi/models/gait_logreg.json` is the file `runtime.py` loads. `pc/models/gait_logreg.joblib` is the sklearn object, for a later experiment on this PC. The check at the bottom of the cell is the inference stub: the same arithmetic as `GaitModel.score`.


In [ ]:
def split_by_time(rows, train_frac=0.7):
    train, test = [], []
    for label in GAIT_LABELS:
        group = sorted((item for item in rows if item["label"] == label), key=lambda item: item["t1"])
        cut = max(1, min(len(group) - 1, int(round(len(group) * train_frac))))
        cut_time = group[cut - 1]["t1"]
        train.extend(group[:cut])
        held = [item for item in group[cut:] if item["t0"] >= cut_time - 1e-9]
        if not held:
            held = group[cut:]
            print("Note: %s test windows still overlap the training slice." % label)
        test.extend(held)
    return train, test


train_rows, test_rows = split_by_time(labeled)
print("train %d   test %d" % (len(train_rows), len(test_rows)))


def matrix(rows):
    x = np.array([item["features"] for item in rows], dtype=float)
    y = np.array([1 if item["label"] == "walking" else 0 for item in rows], dtype=int)
    return x, y


x_train, y_train = matrix(train_rows)
x_test, y_test = matrix(test_rows)

print("Per-class feature means on the labeled windows:")
for label in GAIT_LABELS:
    block = np.array([item["features"] for item in labeled if item["label"] == label], dtype=float)
    means = block.mean(axis=0)
    print(" ", label, ", ".join("%s=%.3f" % (name, value) for name, value in zip(FEATURE_NAMES, means)))

scaler = StandardScaler()
x_train_s = scaler.fit_transform(x_train)
x_test_s = scaler.transform(x_test)
clf = LogisticRegression(max_iter=1000, solver="lbfgs")
clf.fit(x_train_s, y_train)

train_hat = clf.predict(x_train_s)
test_hat = clf.predict(x_test_s)
print("train accuracy %.3f" % accuracy_score(y_train, train_hat))
print("test accuracy  %.3f" % accuracy_score(y_test, test_hat))
if not PATHS:
    print("These demo traces are easy to separate. Train again on a real collection before wearing the model.")
print("confusion matrix rows are true [still, walking], columns are predicted")
print(confusion_matrix(y_test, test_hat, labels=[0, 1]))
print(classification_report(y_test, test_hat, labels=[0, 1], target_names=["still", "walking"], digits=3))

order = sorted(zip(FEATURE_NAMES, clf.coef_.ravel()), key=lambda pair: abs(pair[1]), reverse=True)
print("Standardized coefficients, largest magnitude first. Positive pushes toward walking.")
for name, weight in order:
    print("  %-12s %+.3f" % (name, weight))

pi_model = GaitModel(
    mean=scaler.mean_.tolist(),
    std=scaler.scale_.tolist(),
    weights=clf.coef_.ravel().tolist(),
    bias=float(clf.intercept_[0]),
    threshold=0.5,
    source="train_gait.ipynb",
)
train_p = clf.predict_proba(x_train_s)[:, 1]
gaps = [abs(pi_model.score(row.tolist()) - p) for row, p in zip(x_train, train_p)]
print("max |Pi score - sklearn proba| on the training rows: %.3e" % max(gaps))
if max(gaps) > 1e-4:
    raise RuntimeError("GaitModel.score does not match LogisticRegression.predict_proba")

model_path = REPO / "pi" / "models" / "gait_logreg.json"
pi_model.save(model_path)
payload = json.loads(model_path.read_text(encoding="utf-8"))
payload["note"] = "Exported by pc/train_gait.ipynb from %s." % (
    "the synthesized demo" if not PATHS else ", ".join(str(p) for p in PATHS)
)
model_path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")

import joblib
joblib_path = REPO / "pc" / "models" / "gait_logreg.joblib"
joblib_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(
    {"model": clf, "scaler": scaler, "features": list(FEATURE_NAMES), "window_s": WINDOW_S},
    joblib_path,
)
print("wrote", model_path)
print("wrote", joblib_path)
print((REPO / "pi" / "models" / "INPUTS.md").read_text(encoding="utf-8"))


## Windows and the decision

The first figure is one still window and one walking window, as acceleration magnitude over the 1.5 s. The second is the Pi model's probability in the two features the prior leans on (`mag_std` and `band_energy`). The other six features are held at the training-set mean, so the curve is a slice of the 8-D boundary, at probability 0.5. The third figure is the confusion matrix of the time-held-out windows, and the bar chart is the standardized coefficient of each feature.


In [ ]:
def magnitude_trace(samples):
    t0 = samples[0][0]
    times = []
    values = []
    for mono, x, y, z in samples:
        times.append(mono - t0)
        values.append(math.sqrt(x * x + y * y + z * z))
    return times, values


still_w = next(item for item in labeled if item["label"] == "still")
walk_w = next(item for item in labeled if item["label"] == "walking")
fig, ax = plt.subplots(figsize=(7, 3.5))
for item, name in ((still_w, "still"), (walk_w, "walking")):
    times, values = magnitude_trace(item["samples"])
    ax.plot(times, values, label="%s  t0=%.1f" % (name, item["t0"]))
ax.set_xlabel("seconds into the window")
ax.set_ylabel("acceleration magnitude (m/s²)")
ax.set_title("Sample 1.5 s windows")
ax.legend()
fig.tight_layout()
plt.show()

i_std = list(FEATURE_NAMES).index("mag_std")
i_band = list(FEATURE_NAMES).index("band_energy")
cloud = np.array([item["features"] for item in labeled], dtype=float)
base = cloud.mean(axis=0)
std_lo, std_hi = float(cloud[:, i_std].min()), float(cloud[:, i_std].max())
band_lo, band_hi = float(cloud[:, i_band].min()), float(cloud[:, i_band].max())
if std_hi - std_lo < 1e-3:
    std_lo, std_hi = std_lo - 0.5, std_hi + 0.5
if band_hi - band_lo < 1e-3:
    band_lo, band_hi = band_lo - 0.5, band_hi + 0.5
grid_std = np.linspace(std_lo, std_hi, 70)
grid_band = np.linspace(band_lo, band_hi, 70)
prob = np.zeros((len(grid_band), len(grid_std)))
for r, band in enumerate(grid_band):
    for c, std in enumerate(grid_std):
        vec = base.copy()
        vec[i_std] = std
        vec[i_band] = band
        prob[r, c] = pi_model.score(vec.tolist())

fig, ax = plt.subplots(figsize=(6.5, 5))
image = ax.contourf(grid_std, grid_band, prob, levels=12, cmap="coolwarm")
ax.contour(grid_std, grid_band, prob, levels=[0.5], colors="black", linewidths=1.5)
for label, marker in (("still", "o"), ("walking", "^")):
    block = np.array([item["features"] for item in labeled if item["label"] == label], dtype=float)
    ax.scatter(block[:, i_std], block[:, i_band], marker=marker, edgecolors="black", label=label)
ax.set_xlabel("mag_std (m/s²)")
ax.set_ylabel("band_energy")
ax.set_title("P(walking), other features held at the training mean")
ax.legend(loc="upper left")
fig.colorbar(image, ax=ax, label="P(walking)")
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
cm = confusion_matrix(y_test, test_hat, labels=[0, 1])
axes[0].imshow(cm, cmap="Blues")
axes[0].set_xticks([0, 1], ["still", "walking"])
axes[0].set_yticks([0, 1], ["still", "walking"])
axes[0].set_xlabel("predicted")
axes[0].set_ylabel("true")
axes[0].set_title("Test windows")
for r in range(2):
    for c in range(2):
        axes[0].text(c, r, str(cm[r, c]), ha="center", va="center")
names = [name for name, _weight in order][::-1]
weights = [weight for _name, weight in order][::-1]
colors = ["#b34040" if weight > 0 else "#3d5a80" for weight in weights]
axes[1].barh(names, weights, color=colors)
axes[1].axvline(0.0, color="black", linewidth=0.6)
axes[1].set_xlabel("standardized coefficient")
axes[1].set_title("Positive = walking")
fig.tight_layout()
plt.show()
